In [2]:
import pandas as pd
import lightgbm as lgb

train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/val.csv")

CAT_COLS = ["main_category", "sub_category", "sub_sub_category", "condition_label"]
NUM_COLS = ["item_condition_id", "shipping", "category_depth", "name_length",
            "desc_length", "name_word_count", "has_description", "is_branded",
            "cat_avg_price", "brand_avg_price"]

for c in CAT_COLS:
    train[c] = train[c].fillna("missing").astype("category")
    val[c] = val[c].fillna("missing").astype("category")

X_train = train[NUM_COLS + CAT_COLS]
X_val = val[NUM_COLS + CAT_COLS]
y_train = train["log_price"]
y_val = val["log_price"]

print(X_train.dtypes)
print("Shape:", X_train.shape, X_val.shape)

item_condition_id       int64
shipping                int64
category_depth          int64
name_length             int64
desc_length             int64
name_word_count         int64
has_description         int64
is_branded              int64
cat_avg_price         float64
brand_avg_price       float64
main_category        category
sub_category         category
sub_sub_category     category
condition_label      category
dtype: object
Shape: (39780, 14) (9945, 14)


In [3]:
import numpy as np
from sklearn.metrics import mean_squared_error

model_lgb = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1
)

model_lgb.fit(X_train, y_train)
pred_log_lgb = model_lgb.predict(X_val)
rmsle_lgb = np.sqrt(mean_squared_error(y_val, pred_log_lgb))

print("LightGBM val RMSLE:", round(rmsle_lgb, 4))
print("Ridge val RMSLE", 0.5204)

LightGBM val RMSLE: 0.5677
Ridge val RMSLE 0.5204


In [4]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
import scipy.sparse as sp

scaler_struct = StandardScaler()
X_num_train_struct = scaler_struct.fit_transform(train[NUM_COLS])
X_num_val_struct = scaler_struct.transform(val[NUM_COLS])

ohe_struct = OneHotEncoder(handle_unknown="ignore")
X_cat_train_struct = ohe_struct.fit_transform(train[CAT_COLS])
X_cat_val_struct = ohe_struct.transform(val[CAT_COLS])

X_train_struct = sp.hstack([X_num_train_struct, X_cat_train_struct]).tocsr()
X_val_struct = sp.hstack([X_num_val_struct, X_cat_val_struct]).tocsr()

ridge_struct = Ridge(alpha=0.5, random_state=42)
ridge_struct.fit(X_train_struct, y_train)

pred_log_ridge_struct = ridge_struct.predict(X_val_struct)
rmsle_ridge_struct = np.sqrt(mean_squared_error(y_val, pred_log_ridge_struct))

print("Ridge (structured only, no text)  val RMSLE:", round(rmsle_ridge_struct, 4))
print("LightGBM (structured only)        val RMSLE:", round(rmsle_lgb, 4))
print("Ridge (structured + text)  val RMSLE:", 0.5204)

Ridge (structured only, no text)  val RMSLE: 0.5983
LightGBM (structured only)        val RMSLE: 0.5677
Ridge (structured + text)  val RMSLE: 0.5204


In [6]:
import xgboost as xgb
import numpy as np
from sklearn.metrics import mean_squared_error

for c in CAT_COLS:
    train[c] = train[c].fillna("missing").astype("category")
    cat_dtype = pd.CategoricalDtype(categories=train[c].cat.categories)
    val[c] = val[c].fillna("missing").astype(cat_dtype)

X_train = train[NUM_COLS + CAT_COLS]
X_val = val[NUM_COLS + CAT_COLS]

model_xgb = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    enable_categorical=True,
    tree_method="hist"
)
model_xgb.fit(X_train, y_train)
pred_log_xgb = model_xgb.predict(X_val)
rmsle_xgb = np.sqrt(mean_squared_error(y_val, pred_log_xgb))

print("XGBoost (structured only)   val RMSLE:", round(rmsle_xgb, 4))
print("LightGBM (structured only)  val RMSLE:", round(rmsle_lgb, 4))
print("Ridge (structured only)     val RMSLE:", round(rmsle_ridge_struct, 4))

C:\Users\hp\AppData\Local\Temp\ipykernel_16940\3849016265.py:8: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  val[c] = val[c].fillna("missing").astype(cat_dtype)
C:\Users\hp\AppData\Local\Temp\ipykernel_16940\3849016265.py:8: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  val[c] = val[c].fillna("missing").astype(cat_dtype)


XGBoost (structured only)   val RMSLE: 0.5753
LightGBM (structured only)  val RMSLE: 0.5677
Ridge (structured only)     val RMSLE: 0.5983
